In [ ]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.4 MB/s eta 0:00:00


In [ ]:
import optuna
print(optuna.__version__)

5.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone -b orator-baseline-lightgbm https://github.com/Break-Through-Tech/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data.git

Cloning into 'Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 217 (delta 36), reused 42 (delta 33), pack-reused 154 (from 1)
Receiving objects: 100% (217/217), 191.73 KiB | 6.18 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [ ]:
%cd /content/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data
%pip install -e .


/content/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data
Obtaining file:///content/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fraud (pyproject.toml) ... done
  Created wheel for fraud: filename=fraud-0.1.0-0.editable-py3-none-any.whl size=1174 sha256=25242c986e39e0f8be0333e470502186948ca6d79372600cfa614771ba9e02c4
  Stored in directory: /tmp/pip-ephem-wheel-cache-drgvr7gj/wheels/eb/d2/60/addf2289afee0aaa382e597263252e5f1d62dfc3c64292f23e
Successfully built fraud


In [ ]:
import sys

sys.path.insert(
    0,
    "/content/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data/src"
)

print(sys.path[0])

/content/Banking-1A-online-payment-fraud-detection-using-high-dimensional-transaction-data/src


In [ ]:
from fraud.models.lightgbm_baseline import (
    load_merged,
    assign_split,
    prepare_native_features,
    train_lightgbm,
)

print("LightGBM Task #10 setup imports successfully.")

LightGBM Task #10 setup imports successfully.


In [ ]:
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/ieee-fraud-detection (1)")

print((RAW_DIR / "train_transaction.csv").exists())
print((RAW_DIR / "train_identity.csv").exists())


True
True


In [ ]:
df = load_merged(raw_dir=RAW_DIR)

print("Merged shape:", df.shape)
print("Fraud rate:", df["isFraud"].mean())

Merged shape: (590540, 435)
Fraud rate: 0.03499000914417313


In [ ]:
split = assign_split(df)

print("Split source:", split.attrs.get("source"))
print(split.value_counts())

Split source: derived:TransactionDT
train    420066
test      85430
valid     85044
Name: count, dtype: int64


In [ ]:
# Build the same native-categorical train and validation feature sets as Task #4.

train_df = df.loc[split == "train"].copy()
valid_df = df.loc[split == "valid"].copy()

(
    X_train,
    X_others,
    feature_cols,
    numeric_cols,
    categorical_cols,
) = prepare_native_features(
    train_df,
    [valid_df],
)

X_valid = X_others[0]

y_train = train_df["isFraud"].to_numpy()
y_valid = valid_df["isFraud"].to_numpy()

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Numeric features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

Training shape: (420066, 432)
Validation shape: (85044, 432)
Numeric features: 383
Categorical features: 49


In [ ]:
# Prepare LightGBM datasets and calculate the fraud class weight used during tuning.

import lightgbm as lgb

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count
scale_pos_weight = negative_count / positive_count

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=categorical_cols,
    free_raw_data=False,
)

valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_cols,
    reference=train_set,
    free_raw_data=False,
)

print("Positive training cases:", int(positive_count))
print("Negative training cases:", int(negative_count))
print("scale_pos_weight:", round(scale_pos_weight, 3))

Positive training cases: 14785
Negative training cases: 405281
scale_pos_weight: 27.412


In [ ]:
# Define the Optuna search space and score each trial using validation AUPRC.

import optuna

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "verbosity": -1,
        "force_col_wise": True,
        "seed": 42,
        "feature_fraction_seed": 42,
        "bagging_seed": 42,
        "scale_pos_weight": scale_pos_weight,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.10, log=True
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 64, 384, step=64
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 50, 250, step=25
        ),
        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.4, 0.9
        ),
        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.6, 1.0
        ),
        "bagging_freq": 1,
        "lambda_l1": trial.suggest_float(
            "lambda_l1", 1e-4, 10.0, log=True
        ),
        "lambda_l2": trial.suggest_float(
            "lambda_l2", 1e-4, 10.0, log=True
        ),
    }

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
        ],
    )

    return model.best_score["valid"]["average_precision"]


print("Optuna objective defined successfully.")

Optuna objective defined successfully.


In [ ]:
# Run one Optuna trial to verify the tuning pipeline works correctly.

study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=1,
)

print("Trial finished.")
print("AUPRC:", study.best_value)
print("Parameters:", study.best_params)

[I 2026-10-17 18:36:23,520] A new study created in memory with name: no-name-2aa80ab4-c10f-4b69-b68f-857d0d628663
[I 2026-10-17 18:53:21,602] Trial 0 finished with value: 0.631193084514728 and parameters: {'learning_rate': 0.035537253090132094, 'num_leaves': 128, 'min_child_samples': 50, 'feature_fraction': 0.5106580129923906, 'bagging_fraction': 0.7923900820508076, 'lambda_l1': 4.384112554547565, 'lambda_l2': 0.005817418121504799}. Best is trial 0 with value: 0.631193084514728.


Trial finished.
AUPRC: 0.631193084514728
Parameters: {'learning_rate': 0.035537253090132094, 'num_leaves': 128, 'min_child_samples': 50, 'feature_fraction': 0.5106580129923906, 'bagging_fraction': 0.7923900820508076, 'lambda_l1': 4.384112554547565, 'lambda_l2': 0.005817418121504799}


In [ ]:
# Create a smaller stratified training sample so Optuna trials run much faster.

from sklearn.model_selection import train_test_split

X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=150_000,
    stratify=y_train,
    random_state=42,
)

print("Tuning rows:", len(X_tune))
print("Tuning fraud rate:", y_tune.mean())
print("Full training fraud rate:", y_train.mean())

Tuning rows: 150000
Tuning fraud rate: 0.0352
Full training fraud rate: 0.03519685001880657


In [ ]:
# Build the smaller LightGBM training dataset used for faster Optuna trials.

tune_positive = y_tune.sum()
tune_negative = len(y_tune) - tune_positive
tune_scale_pos_weight = tune_negative / tune_positive

tune_train_set = lgb.Dataset(
    X_tune,
    label=y_tune,
    categorical_feature=categorical_cols,
    free_raw_data=False,
)

print("Tuning rows:", len(X_tune))
print("Tuning positives:", int(tune_positive))
print("Tuning scale_pos_weight:", round(tune_scale_pos_weight, 3))

Tuning rows: 150000
Tuning positives: 5280
Tuning scale_pos_weight: 27.409


In [ ]:
# Fix the Optuna objective by creating matching LightGBM datasets inside each trial.

def objective_fast_fixed(trial):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "verbosity": -1,
        "force_col_wise": True,
        "seed": 42,
        "feature_fraction_seed": 42,
        "bagging_seed": 42,
        "scale_pos_weight": tune_scale_pos_weight,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.10, log=True
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 64, 384, step=64
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 50, 250, step=25
        ),
        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.4, 0.9
        ),
        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.6, 1.0
        ),
        "bagging_freq": 1,
        "lambda_l1": trial.suggest_float(
            "lambda_l1", 1e-4, 10.0, log=True
        ),
        "lambda_l2": trial.suggest_float(
            "lambda_l2", 1e-4, 10.0, log=True
        ),
    }

    trial_train_set = lgb.Dataset(
        X_tune,
        label=y_tune,
        categorical_feature=categorical_cols,
        free_raw_data=False,
    )

    trial_valid_set = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=categorical_cols,
        reference=trial_train_set,
        free_raw_data=False,
    )

    model = lgb.train(
        params,
        trial_train_set,
        num_boost_round=1500,
        valid_sets=[trial_valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(75, verbose=False),
        ],
    )

    trial.set_user_attr(
        "best_iteration",
        model.best_iteration,
    )

    return model.best_score["valid"]["average_precision"]


print("Fixed Optuna objective ready.")

Fixed Optuna objective ready.


In [ ]:
# Run 5 Optuna trials and save the study to Drive so tuning progress is preserved.

from pathlib import Path

OPTUNA_DIR = Path("/content/drive/MyDrive/Banking-1A-results")
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

storage_url = (
    "sqlite:////content/drive/MyDrive/"
    "Banking-1A-results/optuna_lightgbm_task10.db"
)

fast_study = optuna.create_study(
    study_name="lightgbm_task10_fast",
    direction="maximize",
    storage=storage_url,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
)

fast_study.optimize(
    objective_fast,
    n_trials=5,
)

print("Completed trials:", len(fast_study.trials))
print("Best AUPRC:", fast_study.best_value)
print("Best parameters:")
print(fast_study.best_params)

[I 2026-10-17 18:57:41,238] A new study created in RDB with name: lightgbm_task10_fast
[I 2026-10-17 19:03:59,106] Trial 0 finished with value: 0.5993867187612401 and parameters: {'learning_rate': 0.03654452355213248, 'num_leaves': 384, 'min_child_samples': 200, 'feature_fraction': 0.6993292420985183, 'bagging_fraction': 0.6624074561769746, 'lambda_l1': 0.000602521573620386, 'lambda_l2': 0.00019517224641449495}. Best is trial 0 with value: 0.5993867187612401.
[I 2026-10-17 19:07:11,417] Trial 1 finished with value: 0.5926555251518205 and parameters: {'learning_rate': 0.0806234057607385, 'num_leaves': 256, 'min_child_samples': 200, 'feature_fraction': 0.41029224714790125, 'bagging_fraction': 0.9879639408647978, 'lambda_l1': 1.452824663751602, 'lambda_l2': 0.0011526449540315614}. Best is trial 0 with value: 0.5993867187612401.
[W 2026-10-17 19:07:18,573] Trial 2 failed with parameters: {'learning_rate': 0.02679909904436604, 'num_leaves': 128, 'min_child_samples': 100, 'feature_fraction':

LightGBMError: Cannot add validation data, since it has different bin mappers with training data

In [ ]:
# Resume the saved study and run 3 additional trials with the fixed objective.

fast_study = optuna.load_study(
    study_name="lightgbm_task10_fast",
    storage=storage_url,
)

fast_study.optimize(
    objective_fast_fixed,
    n_trials=3,
    gc_after_trial=True,
)

completed_trials = [
    trial for trial in fast_study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

print("Successful trials:", len(completed_trials))
print("Best AUPRC:", fast_study.best_value)
print("Best parameters:")
print(fast_study.best_params)
print(
    "Best iteration:",
    fast_study.best_trial.user_attrs.get("best_iteration"),
)

[I 2026-10-17 19:43:05,366] Trial 3 finished with value: 0.6066852385163364 and parameters: {'learning_rate': 0.02448211139618897, 'num_leaves': 128, 'min_child_samples': 125, 'feature_fraction': 0.579246851751961, 'bagging_fraction': 0.8374890013617071, 'lambda_l1': 0.0021351933557705437, 'lambda_l2': 0.33660856023116015}. Best is trial 3 with value: 0.6066852385163364.
[I 2026-10-17 19:51:45,893] Trial 4 finished with value: 0.6074892085353553 and parameters: {'learning_rate': 0.026693708081246957, 'num_leaves': 320, 'min_child_samples': 75, 'feature_fraction': 0.6983829289454246, 'bagging_fraction': 0.7864224285336826, 'lambda_l1': 0.005808899967942987, 'lambda_l2': 1.3330051355686812}. Best is trial 4 with value: 0.6074892085353553.
[I 2026-10-17 19:58:07,050] Trial 5 finished with value: 0.5981378554476705 and parameters: {'learning_rate': 0.02488706181840053, 'num_leaves': 320, 'min_child_samples': 150, 'feature_fraction': 0.4590630509365883, 'bagging_fraction': 0.786772066373294

Successful trials: 5
Best AUPRC: 0.6074892085353553
Best parameters:
{'learning_rate': 0.026693708081246957, 'num_leaves': 320, 'min_child_samples': 75, 'feature_fraction': 0.6983829289454246, 'bagging_fraction': 0.7864224285336826, 'lambda_l1': 0.005808899967942987, 'lambda_l2': 1.3330051355686812}
Best iteration: 1471


In [ ]:
# Evaluate the original Task #4 LightGBM settings on the same 150k tuning sample.

baseline_trial_params = {
    "learning_rate": 0.05,
    "num_leaves": 256,
    "min_child_samples": 100,
    "feature_fraction": 0.5,
    "bagging_fraction": 0.8,
    "lambda_l1": 0.0001,
    "lambda_l2": 1.0,
}

fast_study.enqueue_trial(baseline_trial_params)

fast_study.optimize(
    objective_fast_fixed,
    n_trials=1,
    gc_after_trial=True,
)

print("Best sampled AUPRC:", fast_study.best_value)
print("Best parameters:")
print(fast_study.best_params)
print(
    "Best iteration:",
    fast_study.best_trial.user_attrs.get("best_iteration"),
)

[I 2026-10-17 20:08:12,808] Trial 6 finished with value: 0.6032233628335707 and parameters: {'learning_rate': 0.05, 'num_leaves': 256, 'min_child_samples': 100, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'lambda_l1': 0.0001, 'lambda_l2': 1.0}. Best is trial 4 with value: 0.6074892085353553.


Best sampled AUPRC: 0.6074892085353553
Best parameters:
{'learning_rate': 0.026693708081246957, 'num_leaves': 320, 'min_child_samples': 75, 'feature_fraction': 0.6983829289454246, 'bagging_fraction': 0.7864224285336826, 'lambda_l1': 0.005808899967942987, 'lambda_l2': 1.3330051355686812}
Best iteration: 1471


In [ ]:
# Retrain Optuna's best LightGBM configuration on the full training set.

best_params = {
    "objective": "binary",
    "metric": "average_precision",
    "verbosity": -1,
    "force_col_wise": True,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "scale_pos_weight": scale_pos_weight,
    "bagging_freq": 1,
    **fast_study.best_params,
}

full_train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=categorical_cols,
    free_raw_data=False,
)

full_valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_cols,
    reference=full_train_set,
    free_raw_data=False,
)

tuned_model = lgb.train(
    best_params,
    full_train_set,
    num_boost_round=5000,
    valid_sets=[full_valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(200),
        lgb.log_evaluation(100),
    ],
)

print("Best iteration:", tuned_model.best_iteration)
print(
    "Best validation AUPRC:",
    tuned_model.best_score["valid"]["average_precision"],
)

Training until validation scores don't improve for 200 rounds
[100]	valid's average_precision: 0.537675
[200]	valid's average_precision: 0.587011
[300]	valid's average_precision: 0.610938
[400]	valid's average_precision: 0.62016
[500]	valid's average_precision: 0.628288
[600]	valid's average_precision: 0.633575
[700]	valid's average_precision: 0.637072
[800]	valid's average_precision: 0.641148
[900]	valid's average_precision: 0.644081
[1000]	valid's average_precision: 0.646925
[1100]	valid's average_precision: 0.649077
[1200]	valid's average_precision: 0.651056
[1300]	valid's average_precision: 0.652257
[1400]	valid's average_precision: 0.654109
[1500]	valid's average_precision: 0.655372
[1600]	valid's average_precision: 0.656274
[1700]	valid's average_precision: 0.657239
[1800]	valid's average_precision: 0.658186
[1900]	valid's average_precision: 0.658808
[2000]	valid's average_precision: 0.659159
[2100]	valid's average_precision: 0.659666
[2200]	valid's average_precision: 0.660386
[2

In [ ]:
# Save the tuned LightGBM model and its main Task #10 results to Google Drive.

import json
from pathlib import Path

TUNED_DIR = Path(
    "/content/drive/MyDrive/Banking-1A-results/lightgbm_task10_tuned"
)
TUNED_DIR.mkdir(parents=True, exist_ok=True)

model_path = TUNED_DIR / "lightgbm_task10_tuned.txt"
summary_path = TUNED_DIR / "run_summary.json"

tuned_model.save_model(
    str(model_path),
    num_iteration=tuned_model.best_iteration,
)

run_summary = {
    "task": "#10 LightGBM hyperparameter tuning",
    "model": "lightgbm",
    "train_rows": len(X_train),
    "validation_rows": len(X_valid),
    "n_features": len(feature_cols),
    "n_numeric": len(numeric_cols),
    "n_categorical": len(categorical_cols),
    "baseline_auprc": 0.6620552457417546,
    "tuned_auprc": 0.665335487014643,
    "improvement": 0.665335487014643 - 0.6620552457417546,
    "best_iteration": tuned_model.best_iteration,
    "best_params": fast_study.best_params,
}

with open(summary_path, "w") as f:
    json.dump(run_summary, f, indent=2)

print("Model saved to:", model_path)
print("Summary saved to:", summary_path)

Model saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_tuned/lightgbm_task10_tuned.txt
Summary saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_tuned/run_summary.json


In [ ]:
# Evaluate the tuned model using the same fraud metrics as the Task #4 baseline.

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)

tuned_scores = tuned_model.predict(
    X_valid,
    num_iteration=tuned_model.best_iteration,
)

y_pred = (tuned_scores >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_valid,
    y_pred,
    labels=[0, 1],
).ravel()

fpr_curve, tpr_curve, _ = roc_curve(
    y_valid,
    tuned_scores,
)

def recall_at_review_rate(y_true, y_score, rate):
    order = np.argsort(-y_score)
    k = max(1, int(len(y_score) * rate))
    reviewed = order[:k]

    return float(
        y_true[reviewed].sum() / y_true.sum()
    )

tuned_metrics = {
    "AUPRC": average_precision_score(
        y_valid,
        tuned_scores,
    ),
    "Precision": precision_score(
        y_valid,
        y_pred,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_valid,
        y_pred,
        zero_division=0,
    ),
    "F1": f1_score(
        y_valid,
        y_pred,
        zero_division=0,
    ),
    "FPR": fp / (fp + tn),
    "Recall@1%": recall_at_review_rate(
        y_valid,
        tuned_scores,
        0.01,
    ),
    "Recall@5%": recall_at_review_rate(
        y_valid,
        tuned_scores,
        0.05,
    ),
    "KS": np.max(
        np.abs(tpr_curve - fpr_curve)
    ),
}

for metric, value in tuned_metrics.items():
    print(f"{metric}: {value:.6f}")

AUPRC: 0.665335
Precision: 0.831541
Recall: 0.482663
F1: 0.610794
FPR: 0.003432
Recall@1%: 0.282247
Recall@5%: 0.688627
KS: 0.729752


In [ ]:
# Save the tuned validation metrics and predictions before starting another Optuna round.

import json
import pandas as pd

metrics_path = TUNED_DIR / "evaluation_metrics.json"
predictions_path = TUNED_DIR / "validation_predictions.csv"

with open(metrics_path, "w") as f:
    json.dump(
        {k: float(v) for k, v in tuned_metrics.items()},
        f,
        indent=2,
    )

pd.DataFrame({
    "y_true": y_valid,
    "y_score": tuned_scores,
}).to_csv(
    predictions_path,
    index=False,
)

print("Metrics saved to:", metrics_path)
print("Predictions saved to:", predictions_path)

Metrics saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_tuned/evaluation_metrics.json
Predictions saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_tuned/validation_predictions.csv


In [ ]:
# Define a focused Optuna search around the strongest settings from Round 1.

def objective_refined(trial):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "verbosity": -1,
        "force_col_wise": True,
        "seed": 42,
        "feature_fraction_seed": 42,
        "bagging_seed": 42,
        "scale_pos_weight": tune_scale_pos_weight,
        "bagging_freq": 1,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.02, 0.05
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 192, 384, step=32
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 50, 125, step=25
        ),
        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.60, 0.80
        ),
        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.70, 0.90
        ),
        "lambda_l1": trial.suggest_float(
            "lambda_l1", 1e-4, 0.1, log=True
        ),
        "lambda_l2": trial.suggest_float(
            "lambda_l2", 0.3, 3.0, log=True
        ),
    }

    trial_train_set = lgb.Dataset(
        X_tune,
        label=y_tune,
        categorical_feature=categorical_cols,
        free_raw_data=False,
    )

    trial_valid_set = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=categorical_cols,
        reference=trial_train_set,
        free_raw_data=False,
    )

    model = lgb.train(
        params,
        trial_train_set,
        num_boost_round=2500,
        valid_sets=[trial_valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
        ],
    )

    trial.set_user_attr(
        "best_iteration",
        model.best_iteration,
    )

    return model.best_score["valid"]["average_precision"]


print("Refined Optuna objective ready.")

Refined Optuna objective ready.


In [ ]:
# Run the first 5 trials of the focused Optuna search and save them to Drive.

refined_storage_url = (
    "sqlite:////content/drive/MyDrive/"
    "Banking-1A-results/optuna_lightgbm_task10_refined.db"
)

refined_study = optuna.create_study(
    study_name="lightgbm_task10_refined",
    direction="maximize",
    storage=refined_storage_url,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
)

refined_study.optimize(
    objective_refined,
    n_trials=5,
    gc_after_trial=True,
)

print("Successful trials:", len([
    t for t in refined_study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]))

print("Best sampled AUPRC:", refined_study.best_value)
print("Best parameters:")
print(refined_study.best_params)

print(
    "Best iteration:",
    refined_study.best_trial.user_attrs.get("best_iteration"),
)

[I 2026-10-17 21:25:04,057] A new study created in RDB with name: lightgbm_task10_refined
[I 2026-10-17 21:34:32,593] Trial 0 finished with value: 0.6057932868545824 and parameters: {'learning_rate': 0.031236203565420874, 'num_leaves': 384, 'min_child_samples': 100, 'feature_fraction': 0.7197316968394073, 'bagging_fraction': 0.7312037280884873, 'lambda_l1': 0.00029375384576328325, 'lambda_l2': 0.34292951628939655}. Best is trial 0 with value: 0.6057932868545824.
[I 2026-10-17 21:41:09,527] Trial 1 finished with value: 0.6120105739597035 and parameters: {'learning_rate': 0.04598528437324806, 'num_leaves': 320, 'min_child_samples': 100, 'feature_fraction': 0.6041168988591604, 'bagging_fraction': 0.8939819704323989, 'lambda_l1': 0.03142880890840111, 'lambda_l2': 0.48917062038664405}. Best is trial 1 with value: 0.6120105739597035.
[I 2026-10-17 21:53:47,077] Trial 2 finished with value: 0.6082091386827685 and parameters: {'learning_rate': 0.02545474901621302, 'num_leaves': 224, 'min_child

Successful trials: 5
Best sampled AUPRC: 0.6163974161363506
Best parameters:
{'learning_rate': 0.024184815819561258, 'num_leaves': 256, 'min_child_samples': 75, 'feature_fraction': 0.6912139968434072, 'bagging_fraction': 0.8570351922786027, 'lambda_l1': 0.0003972110727381913, 'lambda_l2': 0.9802925297345285}
Best iteration: 2497


In [ ]:
# Run 5 more refined Optuna trials before selecting the final configuration.

refined_study.optimize(
    objective_refined,
    n_trials=5,
    gc_after_trial=True,
)

completed_refined = [
    t for t in refined_study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]

print("Successful refined trials:", len(completed_refined))
print("Best sampled AUPRC:", refined_study.best_value)
print("Best parameters:")
print(refined_study.best_params)
print(
    "Best iteration:",
    refined_study.best_trial.user_attrs.get("best_iteration"),
)

[I 2026-10-17 22:46:36,737] Trial 5 finished with value: 0.6083893617153268 and parameters: {'learning_rate': 0.04425192044349384, 'num_leaves': 256, 'min_child_samples': 50, 'feature_fraction': 0.7368466053024314, 'bagging_fraction': 0.7880304987479203, 'lambda_l1': 0.00023233503515390126, 'lambda_l2': 0.9382059110341111}. Best is trial 3 with value: 0.6163974161363506.
[I 2026-10-17 22:58:48,543] Trial 6 finished with value: 0.6096689589904475 and parameters: {'learning_rate': 0.02103165563345655, 'num_leaves': 384, 'min_child_samples': 75, 'feature_fraction': 0.7325044568707963, 'bagging_fraction': 0.7623422152178821, 'lambda_l1': 0.0036324869566766076, 'lambda_l2': 1.0564076416403603}. Best is trial 3 with value: 0.6163974161363506.
[I 2026-10-17 23:12:28,613] Trial 7 finished with value: 0.6121031311875615 and parameters: {'learning_rate': 0.025545633665765812, 'num_leaves': 384, 'min_child_samples': 125, 'feature_fraction': 0.7878997883128378, 'bagging_fraction': 0.87896547008552

Successful refined trials: 10
Best sampled AUPRC: 0.6163974161363506
Best parameters:
{'learning_rate': 0.024184815819561258, 'num_leaves': 256, 'min_child_samples': 75, 'feature_fraction': 0.6912139968434072, 'bagging_fraction': 0.8570351922786027, 'lambda_l1': 0.0003972110727381913, 'lambda_l2': 0.9802925297345285}
Best iteration: 2497


In [ ]:
# Train the final refined LightGBM model on the full training set.

refined_best_params = {
    "objective": "binary",
    "metric": "average_precision",
    "verbosity": -1,
    "force_col_wise": True,
    "seed": 42,
    "feature_fraction_seed": 42,
    "bagging_seed": 42,
    "scale_pos_weight": scale_pos_weight,
    "bagging_freq": 1,
    **refined_study.best_params,
}

final_refined_model = lgb.train(
    refined_best_params,
    full_train_set,
    num_boost_round=6000,
    valid_sets=[full_valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(250),
        lgb.log_evaluation(100),
    ],
)

print("Best iteration:", final_refined_model.best_iteration)
print(
    "Best validation AUPRC:",
    final_refined_model.best_score["valid"]["average_precision"],
)

Training until validation scores don't improve for 250 rounds
[100]	valid's average_precision: 0.509778
[200]	valid's average_precision: 0.56284
[300]	valid's average_precision: 0.596213
[400]	valid's average_precision: 0.610822
[500]	valid's average_precision: 0.619808
[600]	valid's average_precision: 0.626542
[700]	valid's average_precision: 0.630867
[800]	valid's average_precision: 0.634836
[900]	valid's average_precision: 0.637974
[1000]	valid's average_precision: 0.640608
[1100]	valid's average_precision: 0.642998
[1200]	valid's average_precision: 0.645745
[1300]	valid's average_precision: 0.648114
[1400]	valid's average_precision: 0.649917
[1500]	valid's average_precision: 0.651644
[1600]	valid's average_precision: 0.652776
[1700]	valid's average_precision: 0.654054
[1800]	valid's average_precision: 0.655522
[1900]	valid's average_precision: 0.656521
[2000]	valid's average_precision: 0.657848
[2100]	valid's average_precision: 0.658741
[2200]	valid's average_precision: 0.659795
[2

In [31]:
# Save the final refined Task #10 model and tuning summary to Google Drive.

import json
from pathlib import Path

FINAL_DIR = Path(
    "/content/drive/MyDrive/Banking-1A-results/lightgbm_task10_final"
)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_model_path = FINAL_DIR / "lightgbm_task10_final.txt"
final_summary_path = FINAL_DIR / "run_summary.json"

final_refined_model.save_model(
    str(final_model_path),
    num_iteration=final_refined_model.best_iteration,
)

final_summary = {
    "task": "#10 LightGBM hyperparameter tuning",
    "model": "lightgbm",
    "train_rows": len(X_train),
    "validation_rows": len(X_valid),
    "n_features": len(feature_cols),
    "n_numeric": len(numeric_cols),
    "n_categorical": len(categorical_cols),
    "baseline_auprc": 0.6620552457417546,
    "first_tuned_auprc": 0.665335487014643,
    "final_tuned_auprc": 0.6683395107525989,
    "improvement_over_baseline": (
        0.6683395107525989 - 0.6620552457417546
    ),
    "best_iteration": final_refined_model.best_iteration,
    "best_params": refined_study.best_params,
    "refined_optuna_trials": 10,
}

with open(final_summary_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print("Model saved to:", final_model_path)
print("Summary saved to:", final_summary_path)

Model saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_final/lightgbm_task10_final.txt
Summary saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_final/run_summary.json


In [32]:
# Evaluate the final refined model using the same fraud metrics as earlier runs.

final_scores = final_refined_model.predict(
    X_valid,
    num_iteration=final_refined_model.best_iteration,
)

final_pred = (final_scores >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_valid,
    final_pred,
    labels=[0, 1],
).ravel()

fpr_curve, tpr_curve, _ = roc_curve(
    y_valid,
    final_scores,
)

final_metrics = {
    "AUPRC": average_precision_score(
        y_valid,
        final_scores,
    ),
    "Precision": precision_score(
        y_valid,
        final_pred,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_valid,
        final_pred,
        zero_division=0,
    ),
    "F1": f1_score(
        y_valid,
        final_pred,
        zero_division=0,
    ),
    "FPR": fp / (fp + tn),
    "Recall@1%": recall_at_review_rate(
        y_valid,
        final_scores,
        0.01,
    ),
    "Recall@5%": recall_at_review_rate(
        y_valid,
        final_scores,
        0.05,
    ),
    "KS": np.max(
        np.abs(tpr_curve - fpr_curve)
    ),
}

for metric, value in final_metrics.items():
    print(f"{metric}: {value:.6f}")

AUPRC: 0.668340
Precision: 0.838998
Recall: 0.487864
F1: 0.616970
FPR: 0.003286
Recall@1%: 0.281207
Recall@5%: 0.697642
KS: 0.728247


In [33]:
# Save the final tuned metrics and compare all three LightGBM experiments.

import json
import pandas as pd

final_metrics_path = FINAL_DIR / "evaluation_metrics.json"
comparison_path = FINAL_DIR / "lightgbm_task10_comparison.csv"

with open(final_metrics_path, "w") as f:
    json.dump(
        {k: float(v) for k, v in final_metrics.items()},
        f,
        indent=2,
    )

comparison = pd.DataFrame([
    {
        "experiment": "Task 4 baseline",
        "AUPRC": 0.662055,
        "Precision": 0.845105,
        "Recall": 0.472954,
        "F1": 0.606492,
        "FPR": 0.003043,
        "Recall@1%": 0.280513,
        "Recall@5%": 0.692788,
        "KS": 0.729681,
    },
    {
        "experiment": "Task 10 first tuned",
        "AUPRC": 0.665335,
        "Precision": 0.831541,
        "Recall": 0.482663,
        "F1": 0.610794,
        "FPR": 0.003432,
        "Recall@1%": 0.282247,
        "Recall@5%": 0.688627,
        "KS": 0.729752,
    },
    {
        "experiment": "Task 10 refined tuned",
        "AUPRC": 0.668340,
        "Precision": 0.838998,
        "Recall": 0.487864,
        "F1": 0.616970,
        "FPR": 0.003286,
        "Recall@1%": 0.281207,
        "Recall@5%": 0.697642,
        "KS": 0.728247,
    },
])

comparison.to_csv(comparison_path, index=False)

display(comparison)

print("Final metrics saved to:", final_metrics_path)
print("Comparison saved to:", comparison_path)

,experiment,AUPRC,Precision,Recall,F1,FPR,Recall@1%,Recall@5%,KS
0,Task 4 baseline,0.662055,0.845105,0.472954,0.606492,0.003043,0.280513,0.692788,0.729681
1,Task 10 first tuned,0.665335,0.831541,0.482663,0.610794,0.003432,0.282247,0.688627,0.729752
2,Task 10 refined tuned,0.668340,0.838998,0.487864,0.616970,0.003286,0.281207,0.697642,0.728247


Final metrics saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_final/evaluation_metrics.json
Comparison saved to: /content/drive/MyDrive/Banking-1A-results/lightgbm_task10_final/lightgbm_task10_comparison.csv
